# TinyLlama HelpSteer2 LoRA Delta Norms

## Purpose

Compare the effective LoRA task-vector norms `|delta_i|` for the five TinyLlama HelpSteer2 specialist adapters before computing the Gram and cosine relationship matrices.

This notebook uses the same effective-update geometry as the relationship-matrix utilities:

`delta_W_l = (lora_alpha / r) * (B_l @ A_l)`

It loads only LoRA adapter configs and adapter weight files on CPU. It does not load TinyLlama, ArmoRM, tokenizer weights, or reward-model weights.

Generated CSV and plot outputs are analysis artifacts and should stay out of git.

## 1. Clone or Update the Repository


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_url = "https://github.com/NZhang137/master-thesis.git"
colab_root = Path("/content") if Path("/content").exists() else Path.cwd()
repo_path = colab_root / "master-thesis"

if (repo_path / ".git").is_dir():
    print(f"Updating repository at {repo_path}")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
else:
    if repo_path.exists():
        shutil.rmtree(repo_path)
    print(f"Cloning repository to {repo_path}")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Working directory: {Path.cwd()}")


## 2. Check Runtime

GPU is not required for this notebook. It only reads LoRA adapter configs and weights on CPU; it does not load TinyLlama or ArmoRM.


In [ ]:
import platform
import shutil
import subprocess

print(f"Python runtime: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print("GPU required: no")

if shutil.which("nvidia-smi"):
    print("GPU detected by runtime, but this notebook will not use it:")
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No GPU detected. That is fine for this CPU-only analysis notebook.")


## 3. Install Lightweight Dependencies

These are only for table handling, plotting, and reading safetensors adapter files. Training/model dependencies such as `transformers`, `peft`, `accelerate`, and `bitsandbytes` are not needed here.


In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas==2.2.2",
        "matplotlib",
        "safetensors",
    ],
    check=True,
)
print("Installed lightweight analysis dependencies.")


## Configuration

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

EXPECTED_ATTRIBUTES = (
    "helpfulness",
    "correctness",
    "coherence",
    "complexity",
    "verbosity",
)

BASE_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
EXPECTED_LORA_RANK = 8
EXPECTED_LORA_ALPHA = 16
EXPECTED_TARGET_MODULES = (
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
)

# Change these if your adapters are stored somewhere else.
# Examples:
#   ADAPTER_SUBDIR = "best-eval-loss"
#   ADAPTER_SUBDIR = "final"
#   ADAPTER_SUBDIR = ""  # adapter weights directly under each adapter directory
ADAPTER_ROOT = Path("adapters")
ADAPTER_DIR_TEMPLATE = "tinyllama-helpsteer2-{attribute}-adapter"
ADAPTER_SUBDIR = "best-eval-loss"

OUTPUT_CSV = Path("results/tinyllama_helpsteer2_geometry/tinyllama_helpsteer2_delta_norms.csv")
OUTPUT_PLOT = Path("results/tinyllama_helpsteer2_geometry/plots/tinyllama_helpsteer2_delta_norms.png")

WARN_SIMILAR_SCALE_PERCENT = 15.0
WARN_NORM_DRIFT_PERCENT = 30.0


def find_project_root(start: Path | None = None) -> tuple[Path, bool]:
    """Find the repository root, with a Colab/standalone fallback."""
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    candidates.extend(
        [
            Path("/content/master-thesis"),
            Path("/content/drive/MyDrive/master-thesis"),
            Path("/content"),
        ]
    )
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate.resolve(), True

    fallback = current
    print(
        "WARNING: Could not find a full repo root containing src/ and notebooks/. "
        f"Using {fallback} as PROJECT_ROOT in standalone mode. "
        "This is OK if you only upload/extract adapter zips here."
    )
    return fallback, False


PROJECT_ROOT, REPO_ROOT_FOUND = find_project_root()
if REPO_ROOT_FOUND and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ADAPTER_ROOT = (PROJECT_ROOT / ADAPTER_ROOT).resolve() if not ADAPTER_ROOT.is_absolute() else ADAPTER_ROOT
OUTPUT_CSV = (PROJECT_ROOT / OUTPUT_CSV).resolve() if not OUTPUT_CSV.is_absolute() else OUTPUT_CSV
OUTPUT_PLOT = (PROJECT_ROOT / OUTPUT_PLOT).resolve() if not OUTPUT_PLOT.is_absolute() else OUTPUT_PLOT

print(f"Project root: {PROJECT_ROOT}")
print(f"Repo utilities available: {REPO_ROOT_FOUND}")
print(f"Base model name recorded for context only: {BASE_MODEL_NAME}")
print(f"Adapter root: {ADAPTER_ROOT}")
print(f"Adapter subdir: {ADAPTER_SUBDIR!r}")
print(f"Expected LoRA rank/alpha: r={EXPECTED_LORA_RANK}, alpha={EXPECTED_LORA_ALPHA}")
print(f"Expected TinyLlama target modules: {', '.join(EXPECTED_TARGET_MODULES)}")

## Optional Adapter ZIP Upload

Run this cell only in Colab when the adapter folders are not already available under `adapters/`. The uploaded zip is extracted locally for analysis; adapter files are not modified.


In [ ]:
import io
import zipfile

RUN_ADAPTER_ZIP_UPLOAD = True

# If your zip contains an "adapters/" folder, keep PROJECT_ROOT.
# If your zip contains the five adapter folders directly, change this to ADAPTER_ROOT.
ADAPTER_ZIP_EXTRACT_ROOT = PROJECT_ROOT


def safe_extract_zip(zip_file: zipfile.ZipFile, extract_root: Path) -> None:
    """Extract a zip file while preventing paths from escaping extract_root."""
    extract_root = extract_root.resolve()
    for member in zip_file.infolist():
        target_path = (extract_root / member.filename).resolve()
        if not str(target_path).startswith(str(extract_root)):
            raise ValueError(f"Unsafe zip member path: {member.filename}")
    zip_file.extractall(extract_root)


if RUN_ADAPTER_ZIP_UPLOAD:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError("Adapter zip upload is only available in Google Colab.") from error

    uploaded_files = files.upload()
    if not uploaded_files:
        raise RuntimeError("No adapter zip was uploaded.")

    ADAPTER_ZIP_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    for uploaded_name, uploaded_bytes in uploaded_files.items():
        if not uploaded_name.lower().endswith(".zip"):
            raise ValueError(f"Expected a .zip file, got: {uploaded_name}")
        with zipfile.ZipFile(io.BytesIO(uploaded_bytes)) as adapter_zip:
            safe_extract_zip(adapter_zip, ADAPTER_ZIP_EXTRACT_ROOT)
        print(f"Extracted {uploaded_name} to {ADAPTER_ZIP_EXTRACT_ROOT}")

    print("Adapter zip extraction complete. Re-run Adapter Discovery below.")
else:
    print("Adapter zip upload skipped. Set RUN_ADAPTER_ZIP_UPLOAD = True to upload in Colab.")


## 4. Show Important Files

This cell confirms the repository utilities, adapter root, adapter candidates, and output paths before the norm computation starts.


In [ ]:
print(f"Current working directory: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Repo utilities available: {REPO_ROOT_FOUND}")
print(f"Adapter root: {ADAPTER_ROOT}")
print(f"Adapter subdir: {ADAPTER_SUBDIR!r}")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"Output plot: {OUTPUT_PLOT}")

important_paths = [
    PROJECT_ROOT / "src" / "effective_lora_geometry.py",
    ADAPTER_ROOT,
    OUTPUT_CSV.parent,
    OUTPUT_PLOT.parent,
]
for important_path in important_paths:
    print(f"{'FOUND' if important_path.exists() else 'MISSING'}: {important_path}")

print("\nAdapter candidates:")
for attribute in EXPECTED_ATTRIBUTES:
    adapter_dir = ADAPTER_ROOT / ADAPTER_DIR_TEMPLATE.format(attribute=attribute)
    adapter_path = adapter_dir if ADAPTER_SUBDIR in {"", ".", None} else adapter_dir / str(ADAPTER_SUBDIR)
    config_path = adapter_path / "adapter_config.json"
    safetensors_path = adapter_path / "adapter_model.safetensors"
    bin_path = adapter_path / "adapter_model.bin"
    print(f"  {attribute:11s}: {adapter_path}")
    print(f"      config: {'yes' if config_path.is_file() else 'no'}")
    print(f"      weights: {'safetensors' if safetensors_path.is_file() else 'bin' if bin_path.is_file() else 'missing'}")


## Adapter Discovery

In [ ]:
def resolve_adapter_path(attribute: str) -> Path:
    """Resolve one adapter path from the configurable adapter root and subdirectory."""
    adapter_dir = ADAPTER_ROOT / ADAPTER_DIR_TEMPLATE.format(attribute=attribute)
    if ADAPTER_SUBDIR in {"", ".", None}:
        return adapter_dir
    return adapter_dir / str(ADAPTER_SUBDIR)


def discover_adapters() -> dict[str, Path]:
    """Discover the five expected HelpSteer2 adapters and fail clearly if any are missing."""
    discovered: dict[str, Path] = {}
    missing: list[tuple[str, Path]] = []

    print("Configured adapter paths:")
    for attribute in EXPECTED_ATTRIBUTES:
        path = resolve_adapter_path(attribute)
        status = "FOUND" if path.is_dir() else "MISSING"
        print(f"  {attribute:11s} {status:7s} {path}")
        if path.is_dir():
            discovered[attribute] = path
        else:
            missing.append((attribute, path))

    if missing:
        details = "\n".join(f"  - {attribute}: {path}" for attribute, path in missing)
        raise FileNotFoundError(
            "Fewer than five TinyLlama HelpSteer2 adapters were found.\n"
            "Missing adapter directories:\n"
            f"{details}\n\n"
            "If your adapters are stored in a different checkpoint subdirectory, "
            "change ADAPTER_SUBDIR above, for example to 'final' or ''."
        )

    if len(discovered) != len(EXPECTED_ATTRIBUTES):
        raise RuntimeError(f"Expected 5 adapters, found {len(discovered)}.")
    return discovered


adapter_paths = discover_adapters()

## LoRA Loading Utilities

In [ ]:
try:
    from src.effective_lora_geometry import (
        effective_lora_inner_product,
        effective_lora_update_norm,
        load_adapter_config,
        load_effective_lora_geometry,
    )
    print("Using repository LoRA geometry utilities from src.effective_lora_geometry.")
except ImportError:
    print("Repository utility import failed; using standalone LoRA geometry utilities in this notebook.")
    import json
    import re
    from collections.abc import Mapping
    from dataclasses import dataclass
    from typing import Any

    import torch

    LORA_WEIGHT_FILENAMES = ("adapter_model.safetensors", "adapter_model.bin")
    LORA_FACTOR_PATTERN = re.compile(
        r"^(?P<module>.+)\.lora_(?P<factor>[AB])(?:\.[^.]+)?\.weight$"
    )

    @dataclass(frozen=True)
    class EffectiveLoraLayer:
        lora_a: torch.Tensor
        lora_b: torch.Tensor
        scaling: float

        @property
        def effective_shape(self) -> tuple[int, int]:
            return (int(self.lora_b.shape[0]), int(self.lora_a.shape[1]))

    def parse_lora_factor_key(name: str) -> tuple[str, str] | None:
        match = LORA_FACTOR_PATTERN.fullmatch(name)
        if match is None:
            return None
        return match.group("module"), match.group("factor")

    def load_adapter_config(adapter_path: str | Path) -> dict[str, Any]:
        path = Path(adapter_path)
        config_path = path / "adapter_config.json"
        if not path.is_dir():
            raise FileNotFoundError(f"Adapter directory not found: {path}")
        if not config_path.is_file():
            raise FileNotFoundError(f"Missing adapter_config.json in {path}")
        with config_path.open("r", encoding="utf-8") as config_file:
            config = json.load(config_file)
        if not isinstance(config, dict):
            raise ValueError(f"Adapter config must contain a JSON object: {config_path}")
        return config

    def load_lora_factor_state_dict(adapter_path: str | Path) -> dict[str, torch.Tensor]:
        path = Path(adapter_path)
        load_adapter_config(path)
        safetensors_path = path / LORA_WEIGHT_FILENAMES[0]
        bin_path = path / LORA_WEIGHT_FILENAMES[1]
        if safetensors_path.is_file():
            try:
                from safetensors.torch import load_file
            except ImportError as error:
                raise ImportError("Install safetensors to read adapter_model.safetensors.") from error
            state_dict = load_file(str(safetensors_path), device="cpu")
        elif bin_path.is_file():
            try:
                state_dict = torch.load(bin_path, map_location="cpu", weights_only=True)
            except TypeError:
                state_dict = torch.load(bin_path, map_location="cpu")
        else:
            expected = " or ".join(LORA_WEIGHT_FILENAMES)
            raise FileNotFoundError(f"Missing {expected} in {path}")
        if not isinstance(state_dict, Mapping):
            raise ValueError(f"Adapter weights must contain a state dictionary: {path}")
        factors = {
            key: tensor.detach().cpu()
            for key, tensor in state_dict.items()
            if parse_lora_factor_key(key) is not None and torch.is_tensor(tensor)
        }
        if not factors:
            raise ValueError(f"No LoRA A/B tensors were found in adapter weights: {path}")
        return factors

    def resolve_pattern_value(pattern: Mapping[str, Any] | None, module_name: str, default: Any) -> Any:
        if not pattern:
            return default
        matches = [
            (key, value)
            for key, value in pattern.items()
            if module_name == key or module_name.endswith(f".{key}")
        ]
        if not matches:
            return default
        return max(matches, key=lambda item: len(item[0]))[1]

    def group_lora_factors(state_dict: Mapping[str, torch.Tensor]) -> dict[str, dict[str, torch.Tensor]]:
        factors: dict[str, dict[str, torch.Tensor]] = {}
        for key, tensor in state_dict.items():
            parsed = parse_lora_factor_key(key)
            if parsed is None:
                continue
            module_name, factor_name = parsed
            if tensor.ndim != 2 or not torch.is_floating_point(tensor):
                raise ValueError(f"LoRA factor must be a floating-point matrix: {key}")
            value = tensor.detach().to(device="cpu", dtype=torch.float64)
            if not torch.isfinite(value).all():
                raise ValueError(f"LoRA factor contains non-finite values: {key}")
            module_factors = factors.setdefault(module_name, {})
            if factor_name in module_factors:
                raise ValueError(f"Duplicate LoRA {factor_name} factor for {module_name}")
            module_factors[factor_name] = value
        if not factors:
            raise ValueError("No LoRA A/B factor pairs were found.")
        for module_name, module_factors in factors.items():
            missing = {"A", "B"}.difference(module_factors)
            if missing:
                raise ValueError(f"Incomplete LoRA factors for {module_name}: missing {sorted(missing)}")
        return factors

    def load_effective_lora_geometry(adapter_path: str | Path) -> dict[str, EffectiveLoraLayer]:
        config = load_adapter_config(adapter_path)
        if config.get("use_dora", False):
            raise ValueError("DoRA requires a different effective-update formula.")
        if config.get("modules_to_save"):
            raise ValueError("Adapters with modules_to_save are not pure LoRA updates and are unsupported.")
        default_rank = int(config["r"])
        default_alpha = float(config["lora_alpha"])
        rank_pattern = config.get("rank_pattern") or {}
        alpha_pattern = config.get("alpha_pattern") or {}
        use_rslora = bool(config.get("use_rslora", False))
        grouped = group_lora_factors(load_lora_factor_state_dict(adapter_path))
        geometry: dict[str, EffectiveLoraLayer] = {}
        for module_name, factors in grouped.items():
            lora_a = factors["A"]
            lora_b = factors["B"]
            actual_rank = int(lora_a.shape[0])
            if lora_b.shape[1] != actual_rank:
                raise ValueError(f"Incompatible LoRA shapes for {module_name}: A={tuple(lora_a.shape)}, B={tuple(lora_b.shape)}")
            configured_rank = int(resolve_pattern_value(rank_pattern, module_name, default_rank))
            if configured_rank != actual_rank:
                raise ValueError(f"Configured rank for {module_name} is {configured_rank}, saved rank is {actual_rank}.")
            alpha = float(resolve_pattern_value(alpha_pattern, module_name, default_alpha))
            denominator = math.sqrt(actual_rank) if use_rslora else actual_rank
            geometry[module_name] = EffectiveLoraLayer(lora_a=lora_a, lora_b=lora_b, scaling=alpha / denominator)
        return geometry

    def effective_lora_inner_product(geometry_a: Mapping[str, EffectiveLoraLayer], geometry_b: Mapping[str, EffectiveLoraLayer]) -> float:
        if sorted(geometry_a) != sorted(geometry_b):
            raise ValueError("Adapters update different LoRA modules.")
        total = torch.zeros((), dtype=torch.float64)
        for module_name in sorted(geometry_a):
            layer_a = geometry_a[module_name]
            layer_b = geometry_b[module_name]
            if layer_a.effective_shape != layer_b.effective_shape:
                raise ValueError(f"Incompatible effective shape for {module_name}.")
            b_gram = layer_a.lora_b.transpose(0, 1) @ layer_b.lora_b
            a_gram = layer_a.lora_a @ layer_b.lora_a.transpose(0, 1)
            total += layer_a.scaling * layer_b.scaling * torch.sum(b_gram * a_gram)
        return float(total.item())

    def effective_lora_update_norm(geometry: Mapping[str, EffectiveLoraLayer], eps: float = 1e-12) -> float:
        squared_norm = effective_lora_inner_product(geometry, geometry)
        if squared_norm < -eps:
            raise ValueError(f"Effective update has invalid squared norm: {squared_norm}")
        norm = math.sqrt(max(squared_norm, 0.0))
        if norm < eps:
            raise ValueError("Effective LoRA update is near zero; cosine is undefined.")
        return norm


def _single_value(values: set[float] | set[int], label: str, adapter_path: Path):
    """Return one value or fail if an adapter uses mixed per-module values."""
    if len(values) != 1:
        raise ValueError(
            f"Expected a uniform {label} for {adapter_path}, got {sorted(values)}. "
            "This notebook expects the TinyLlama training LoRA config with r=8 and alpha=16."
        )
    return next(iter(values))


def summarize_effective_lora_adapter(attribute: str, adapter_path: Path) -> dict[str, object]:
    """Compute the effective LoRA task-vector norm for one adapter.

    The norm is computed from PEFT-scaled effective updates. The repository utility
    uses the equivalent low-rank Frobenius identity instead of materializing every
    full B @ A matrix, so this remains CPU-friendly and does not load the base model.
    """
    config = load_adapter_config(adapter_path)
    geometry = load_effective_lora_geometry(adapter_path)
    if not geometry:
        raise ValueError(f"No effective LoRA modules found for {attribute}: {adapter_path}")

    ranks = {int(layer.lora_a.shape[0]) for layer in geometry.values()}
    scalings = {float(layer.scaling) for layer in geometry.values()}
    rank = int(_single_value(ranks, "LoRA rank", adapter_path))
    scaling = float(_single_value(scalings, "LoRA scaling", adapter_path))

    use_rslora = bool(config.get("use_rslora", False))
    alpha = scaling * (math.sqrt(rank) if use_rslora else rank)

    if rank != EXPECTED_LORA_RANK or not math.isclose(alpha, EXPECTED_LORA_ALPHA):
        raise ValueError(
            f"Unexpected LoRA config for {attribute}: r={rank}, alpha={alpha}. "
            f"Expected r={EXPECTED_LORA_RANK}, alpha={EXPECTED_LORA_ALPHA}."
        )

    configured_targets = set(config.get("target_modules") or [])
    expected_targets = set(EXPECTED_TARGET_MODULES)
    if configured_targets and configured_targets != expected_targets:
        raise ValueError(
            f"Unexpected target_modules for {attribute}: "
            f"{sorted(configured_targets)} != {sorted(expected_targets)}"
        )

    delta_norm_squared = float(effective_lora_inner_product(geometry, geometry))
    delta_norm = float(effective_lora_update_norm(geometry))

    return {
        "attribute": attribute,
        "adapter_path": str(adapter_path),
        "num_lora_modules": len(geometry),
        "lora_rank": rank,
        "lora_alpha": alpha,
        "scaling": scaling,
        "delta_norm": delta_norm,
        "delta_norm_squared": delta_norm_squared,
    }


## Delta Norm Computation

In [ ]:
rows = [
    summarize_effective_lora_adapter(attribute, adapter_paths[attribute])
    for attribute in EXPECTED_ATTRIBUTES
]

results_df = pd.DataFrame(rows)
mean_delta_norm = float(results_df["delta_norm"].mean())
results_df["relative_to_mean"] = results_df["delta_norm"] / mean_delta_norm
results_df["percent_deviation_from_mean"] = (
    100.0 * (results_df["delta_norm"] - mean_delta_norm) / mean_delta_norm
)

print(f"Mean effective LoRA delta norm across adapters: {mean_delta_norm:.8f}")
print("Computed |delta_i| from effective LoRA updates only; TinyLlama/ArmoRM were not loaded.")

## Results Table

In [ ]:
display_columns = [
    "attribute",
    "adapter_path",
    "num_lora_modules",
    "lora_rank",
    "lora_alpha",
    "scaling",
    "delta_norm",
    "delta_norm_squared",
    "relative_to_mean",
    "percent_deviation_from_mean",
]

display(
    results_df[display_columns].style.format(
        {
            "lora_alpha": "{:.6g}",
            "scaling": "{:.6g}",
            "delta_norm": "{:.8f}",
            "delta_norm_squared": "{:.8f}",
            "relative_to_mean": "{:.4f}",
            "percent_deviation_from_mean": "{:+.2f}%",
        }
    )
)

## Norm Drift Interpretation

In [ ]:
def interpret_norm_drift(percent_deviation: float) -> str:
    """Classify norm scale relative to the five-adapter mean."""
    absolute_deviation = abs(percent_deviation)
    if absolute_deviation <= WARN_SIMILAR_SCALE_PERCENT:
        return "acceptable / similar scale"
    if absolute_deviation > WARN_NORM_DRIFT_PERCENT:
        return "WARNING: possible norm drift"
    return "moderate deviation: inspect before using geometry as a proxy"


print(f"Mean |delta|: {mean_delta_norm:.8f}")
print(
    f"Similar-scale band: +/-{WARN_SIMILAR_SCALE_PERCENT:.0f}% of the mean; "
    f"norm-drift warning above {WARN_NORM_DRIFT_PERCENT:.0f}% deviation."
)

for row in results_df.itertuples(index=False):
    message = interpret_norm_drift(row.percent_deviation_from_mean)
    print(
        f"{row.attribute:11s}: |delta|={row.delta_norm:.8f}, "
        f"deviation={row.percent_deviation_from_mean:+.2f}% -> {message}"
    )

if (results_df["percent_deviation_from_mean"].abs() > WARN_NORM_DRIFT_PERCENT).any():
    print(
        "\nWARNING: At least one adapter deviates by more than 30% from the mean. "
        "Check whether training length, checkpoint choice, or adapter export differs."
    )
else:
    print("\nNo adapter exceeds the 30% norm-drift warning threshold.")

## Plot

In [ ]:
OUTPUT_PLOT.parent.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(results_df["attribute"], results_df["delta_norm"], color="#4C78A8")
ax.axhline(mean_delta_norm, color="#D62728", linestyle="--", linewidth=1.5, label="mean |delta|")
ax.set_title("TinyLlama HelpSteer2 Effective LoRA Delta Norms")
ax.set_xlabel("Attribute")
ax.set_ylabel("Effective LoRA task-vector norm |delta_i|")
ax.tick_params(axis="x", rotation=30)
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_PLOT, dpi=200)
plt.show()

print(f"Saved plot to: {OUTPUT_PLOT}")

## Saved Outputs

In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df[display_columns].to_csv(OUTPUT_CSV, index=False)

print(f"Saved CSV to: {OUTPUT_CSV}")
print(f"Saved plot to: {OUTPUT_PLOT}")
print("Note: generated geometry CSVs and plots are analysis artifacts and should stay out of git.")

## Lightweight Validation

In [ ]:
observed_attributes = set(results_df["attribute"])
expected_attributes = set(EXPECTED_ATTRIBUTES)
assert observed_attributes == expected_attributes, (
    f"Attribute mismatch: observed={sorted(observed_attributes)}, "
    f"expected={sorted(expected_attributes)}"
)
assert len(results_df) == len(EXPECTED_ATTRIBUTES), "Expected exactly five adapter rows."
assert (results_df["num_lora_modules"] > 0).all(), "Each adapter must expose at least one LoRA A/B pair."
assert results_df["delta_norm"].map(math.isfinite).all(), "All delta norms must be finite."
assert (results_df["delta_norm"] > 0).all(), "All delta norms must be positive."
assert results_df["delta_norm_squared"].map(math.isfinite).all(), "All squared delta norms must be finite."
assert (results_df["delta_norm_squared"] > 0).all(), "All squared delta norms must be positive."
assert OUTPUT_CSV.is_file(), f"Output CSV was not written: {OUTPUT_CSV}"
assert OUTPUT_PLOT.is_file(), f"Output plot was not written: {OUTPUT_PLOT}"

print("Validation passed:")
print("  - all five expected attributes are present")
print("  - every adapter has LoRA A/B pairs")
print("  - all norms are finite and positive")
print("  - output CSV and plot exist")

## Next Step: Compute Gram and Cosine Relationship Matrices

If the five norms are on a comparable scale, continue with the relationship-matrix step.

The cosine matrix should use the same effective LoRA geometry, not raw concatenated `A` and `B` factors. A Gram matrix can be computed from the same effective-update inner products before cosine normalization:

`G_ij = <delta_i, delta_j>_F`

`R_cos,ij = G_ij / (|delta_i| |delta_j|)`

If one adapter triggers a norm-drift warning, inspect the checkpoint choice and training logs before interpreting relationship matrices.